In [0]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
# Widget definitions (on Databricks) / config (Databricks Connect - widgets API limited)
_scope = "epic_on_fhir_oauth_keys"
_client_key = "client_id"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_algo = "RS384"

try:
    dbutils.widgets.text("epic_secrets_scope", _scope, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("algo", _algo, "Epic Token Enrcyption Algorithm")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

In [0]:
# Databricks Connect doesn't support widgets.getAll(); use config fallback for local dev
try:
    widget_values = dbutils.widgets.getAll()
except AttributeError:
    widget_values = {
        "epic_secrets_scope": _scope,
        "client_id_dbs_key": _client_key,
        "token_url": _token_url,
        "algo": _algo,
    }

for name, value in widget_values.items():
    print(f"{name} = {value}")

In [0]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]
EPIC_KID = dbutils.secrets.get(scope=widget_values["epic_secrets_scope"], key="kid")

In [0]:
# print(f"""
# CLIENT_ID = {CLIENT_ID}
# TOKEN_URL = {TOKEN_URL}
# PRIVATE_KEY = {PRIVATE_KEY}
# ALGO = {ALGO}
# EPIC_KID = {EPIC_KID}
# """)

In [0]:
import sys
import os

# Get the directory containing this notebook
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# Add src directory to path if not already present (notebook is already in src/)
if notebook_dir not in sys.path:
	sys.path.append(notebook_dir)

from smart_on_fhir.auth import EpicApiAuth
from smart_on_fhir.endpoint import EpicApiRequest

In [0]:
epicAuth = EpicApiAuth(
  client_id = CLIENT_ID,
  private_key = PRIVATE_KEY,
  kid = EPIC_KID,
  algo = ALGO,
  auth_location = TOKEN_URL
)

In [0]:
epicAuth.can_connect()

In [0]:
epicFhirAPi = EpicApiRequest(
    auth = epicAuth,
    base_url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/"
)

In [0]:
resource = "Patient"
action = "?identifier=EXTERNAL|Z6129"

In [0]:
patient = epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)

In [0]:
import json

# Display the full response structure
print("Response keys:", patient.keys())
print("\nResponse status:", patient['response']['response_status_code'])
print("\nResponse text (first 500 chars):")
print(patient['response']['response_text'][:500])

In [0]:
# Parse the response text as JSON
patient_data = json.loads(patient['response']['response_text'])

# Display the parsed patient data
display(patient_data)

In [0]:
# Extract the FHIR identifier from the patient resource
identifiers = patient_data['entry'][0]['resource']['identifier']
patient_id = None

for identifier in identifiers:
	if identifier.get('type', {}).get('text') == 'FHIR':
		patient_id = identifier['value']
		break

print(f"Patient FHIR ID: {patient_id}")

In [0]:
patient_id = "erXuFYUfucBZaryVksYEcMg3"

In [0]:
resource = "Patient"
action = f"{patient_id}/$summary"
print(action)

In [0]:
epicFhirAPi.make_request(
    http_method="get",
    resource=resource,
    action=action
)

In [0]:
# Epic requires encounter for Observation create. Search for encounters for our patient.
# Prefer recent encounters (date >= today) so effectiveDateTime "now" falls within encounter period.
# Use 'dt' alias to avoid shadowing datetime module used by EpicApiAuth
from datetime import datetime as dt, timezone

today = dt.now(timezone.utc).strftime("%Y-%m-%d")
enc_resp = epicFhirAPi.make_request(
    http_method="get",
    resource="Encounter",
    action=f"?patient=Patient/{patient_id}&date=le{today}"
)
enc_bundle = json.loads(enc_resp["response"]["response_text"])
encounters = enc_bundle.get("entry", [])

# Fallback: if no recent encounters, use any encounter
if not encounters:
    enc_resp = epicFhirAPi.make_request(http_method="get", resource="Encounter", action=f"?patient=Patient/{patient_id}")
    enc_bundle = json.loads(enc_resp["response"]["response_text"])
    encounters = enc_bundle.get("entry", [])

if not encounters:
    raise ValueError("No encounters found for patient - Observation create requires a valid encounter.")

encounter = encounters[0]["resource"]
encounter_id = encounter["id"]
enc_period = encounter.get("period", {})
enc_start = enc_period.get("start")
enc_end = enc_period.get("end")

In [0]:
# FHIR R4 Observation payload - Heart Rate vital sign (Epic flowsheet code 8)
# Use unique effectiveDateTime to avoid Epic 59189 "Reading already exists"
from datetime import timedelta

now_utc = dt.now(timezone.utc)
if enc_start:
    base = dt.fromisoformat(enc_start.replace("Z", "+00:00")) if "T" in enc_start else dt.fromisoformat(enc_start + "T12:00:00+00:00")
    offset_sec = int(now_utc.timestamp()) % 3600
    effective_dt = (base + timedelta(seconds=offset_sec)).strftime("%Y-%m-%dT%H:%M:%SZ")
else:
    effective_dt = now_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

observation_payload = {
    "resourceType": "Observation",
    "status": "final",
    "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
    "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
    "subject": {"reference": f"Patient/{patient_id}"},
    "encounter": {"reference": f"Encounter/{encounter_id}"},
    "effectiveDateTime": effective_dt,
    "valueQuantity": {"value": 75}
}

In [0]:
obs_resp = epicFhirAPi.make_request(
    http_method="post",
    resource="Observation",
    action="",
    data=json.dumps(observation_payload)
)

# Print OperationOutcome when 4xx/5xx
sc = obs_resp["response"]["response_status_code"]
if sc >= 400:
    try:
        oo = json.loads(obs_resp["response"]["response_text"])
        if oo.get("resourceType") == "OperationOutcome":
            for issue in oo.get("issue", []):
                diag = issue.get("diagnostics", "")
                details = issue.get("details", {}).get("text", "")
                loc = issue.get("location", [])
                print(f"Epic error: {details} | diagnostics: {diag} | location: {loc}")
    except Exception:
        pass
obs_resp

In [0]:
# Extract Observation id from Location header for retrieval
if obs_resp["response"]["response_status_code"] == 201:
    location = obs_resp["response"]["response_headers"].get("Location", "")
    obs_id = location.split("/")[-1] if location else None
else:
    obs_id = None
    print("Observation was not created (status != 201); nothing to retrieve")

In [0]:
try:
    if obs_id:
        new_obs = epicFhirAPi.make_request(
            http_method="get",
            resource="Observation",
            action=obs_id
        )
        if new_obs["response"]["response_status_code"] == 200:
            obs_resource = json.loads(new_obs["response"]["response_text"])
            print(json.dumps(obs_resource, indent=2))
        else:
            print(json.dumps(new_obs, indent=2))
    else:
        print("No Observation id to retrieve (POST did not return 201)")
except Exception as e:
    print(f"Failed to retrieve Observation: {e}")

## AllergyIntolerance Create (Epic API 945)

Create an allergy intolerance for the patient. See [Epic AllergyIntolerance.Create](https://fhir.epic.com/Specifications?api=945).

In [0]:
# FHIR R4 AllergyIntolerance payload - medication allergy (RxNorm Penicillin 7980)
# Epic expects category as array. RxNorm preferred for medications. Recorder/reaction may be required.
# Practitioner eM5CWtq15N0WJeuCet5bJlQ3 = Physician Family Medicine (from Patient.generalPractitioner)
# Epic 59141 = duplicate if allergy already exists.
allergy_payload = {
    "resourceType": "AllergyIntolerance",
    "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
    "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
    "type": "allergy",
    "category": ["medication"],
    "criticality": "low",
    "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
    "patient": {"reference": f"Patient/{patient_id}"},
    "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
    "recordedDate": "2024-01-15",
    "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
}

In [0]:
allergy_resp = epicFhirAPi.make_request(
    http_method="post",
    resource="AllergyIntolerance",
    action="",
    data=json.dumps(allergy_payload)
)

sc = allergy_resp["response"]["response_status_code"]
if sc >= 400:
    try:
        oo = json.loads(allergy_resp["response"]["response_text"])
        if oo.get("resourceType") == "OperationOutcome":
            for issue in oo.get("issue", []):
                diag = issue.get("diagnostics", "")
                details = issue.get("details", {}).get("text", "")
                loc = issue.get("location", [])
                print(f"Epic error: {details} | diagnostics: {diag} | location: {loc}")
    except Exception:
        pass
allergy_resp

In [0]:
# Extract AllergyIntolerance id from Location header for retrieval
if allergy_resp["response"]["response_status_code"] == 201:
    location = allergy_resp["response"]["response_headers"].get("Location", "")
    allergy_id = location.split("/")[-1] if location else None
else:
    allergy_id = None
    print("AllergyIntolerance was not created (status != 201); nothing to retrieve")

In [0]:
try:
    if allergy_id:
        new_allergy = epicFhirAPi.make_request(
            http_method="get",
            resource="AllergyIntolerance",
            action=allergy_id
        )
        if new_allergy["response"]["response_status_code"] == 200:
            allergy_resource = json.loads(new_allergy["response"]["response_text"])
            print(json.dumps(allergy_resource, indent=2))
        else:
            # CaseInsensitiveDict from requests is not JSON serializable — convert headers
            serializable = new_allergy.copy()
            serializable["response"] = new_allergy["response"].copy()
            serializable["response"]["response_headers"] = dict(new_allergy["response"]["response_headers"])
            print(json.dumps(serializable, indent=2))
    else:
        print("No AllergyIntolerance id to retrieve (POST did not return 201)")
except Exception as e:
    print(f"Failed to retrieve AllergyIntolerance: {e}")